# BirdCLEF 2026: Temporal & Migratory EDA

This notebook analyzes the seasonal dynamics of the Pantanal dataset. We contrast **Resident** species (present year-round) with **Migratory** species (seasonal visitors) and analyze how the soundscapes differ across months.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
DATA_DIR = "../data/raw"
sns.set_theme(style="whitegrid")

## 1. Define Migratory vs Resident Species

Based on our previous analysis, we'll identify migratory genera common in the Pantanal.

In [ ]:
taxonomy_df = pd.read_csv(os.path.join(DATA_DIR, "taxonomy.csv"))
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

# List of common migratory genera in the Pantanal
migratory_genera = ['Hirundo', 'Tringa', 'Calidris', 'Tyrannus', 'Progne', 'Pandion', 'Cathartes']

def classify_status(sci_name):
    genus = str(sci_name).split(' ')[0]
    return 'Migratory' if genus in migratory_genera else 'Resident'

taxonomy_df['status'] = taxonomy_df['scientific_name'].apply(classify_status)
status_map = taxonomy_df.set_index('primary_label')['status'].to_dict()

print(f"Target Species Status Distribution:")
print(taxonomy_df['status'].value_counts())

## 2. Temporal Distribution of Soundscapes

Let's extract the month from the 1-minute Pantanal soundscape recordings.

In [ ]:
ss_labels = pd.read_csv(os.path.join(DATA_DIR, "train_soundscapes_labels.csv"))

def extract_month(filename):
    # Format: BC2026_Train_9991_S22_20240213_220000
    try:
        return int(filename.split('_')[4][4:6])
    except:
        return None

ss_labels['month'] = ss_labels['filename'].apply(extract_month)

plt.figure(figsize=(10, 5))
ss_labels['month'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title("Soundscape Windows per Month")
plt.xlabel("Month")
plt.ylabel("Number of 5s Windows")
plt.show()

## 3. Presence of Migrants vs. Residents

How often do migrants actually appear in the soundscapes compared to residents?

In [ ]:
# soundscape labels are semicolon separated lists
def count_status(labels_str):
    if pd.isna(labels_str) or labels_str == "":
        return 0, 0
    
    species_list = labels_str.split(';')
    m_count = sum(1 for s in species_list if status_map.get(s) == 'Migratory')
    r_count = sum(1 for s in species_list if status_map.get(s) == 'Resident')
    return m_count, r_count

counts = ss_labels['primary_label'].apply(count_status)
ss_labels['migrant_count'] = [c[0] for c in counts]
ss_labels['resident_count'] = [c[1] for c in counts]

monthly_stats = ss_labels.groupby('month')[['migrant_count', 'resident_count']].mean()

monthly_stats.plot(kind='line', marker='o', figsize=(12, 6))
plt.title("Average Number of Species Detected per 5s Window")
plt.ylabel("Mean Count")
plt.grid(True)
plt.show()

## 4. Deep Dive: Top Migratory Species Seasonality

Which migratory species are driving the temporal patterns?

In [ ]:
# Flatten labels to see individual species presence
expanded_ss = ss_labels.assign(species=ss_labels['primary_label'].str.split(';')).explode('species')
expanded_ss = expanded_ss[expanded_ss['species'].notna() & (expanded_ss['species'] != "")]
expanded_ss['status'] = expanded_ss['species'].map(status_map)

migrant_only = expanded_ss[expanded_ss['status'] == 'Migratory']
top_migrants = migrant_only['species'].value_counts().head(5).index

plt.figure(figsize=(14, 7))
for bird in top_migrants:
    bird_data = migrant_only[migrant_only['species'] == bird]
    bird_monthly = bird_data['month'].value_counts().sort_index()
    # Normalize by total windows in that month
    total_monthly = ss_labels['month'].value_counts()
    presence_ratio = bird_monthly / total_monthly
    plt.plot(presence_ratio.index, presence_ratio.values, marker='s', label=f"{bird} ({taxonomy_df[taxonomy_df['primary_label']==bird]['common_name'].values[0]})")

plt.title("Presence Probability of Top Migrants across Months")
plt.xlabel("Month")
plt.ylabel("Detection Probability")
plt.legend()
plt.show()

## 5. Summary Findings for Fusion Model

1. **Seasonality is real:** Certain species (e.g., Shorebirds) only appear in soundscapes from specific flood-pulse months.
2. **The Temporal Branch Opportunity:** Feeding `Month` into the model can help it zero in on migratory candidates while maintaining a steady baseline for residents.
3. **Dialect Investigation:** Future work should compare the visual patterns (spectrograms) of these migrants in the clean `train_audio` (often northern recordings) vs the `train_soundscapes` (Pantanal wintering ground) to check for acoustic shifts.